# Module 13: The Rosetta Stone
## RL and Active Inference Are Closer Than You Think

**Spinning Up in Active Inference** | Notebook 13 of 16

---

Throughout this curriculum we have presented Reinforcement Learning and Active Inference as distinct frameworks. In this module we reveal their deep formal connections. The two are not rival theories -- they are different *decompositions* of the same underlying problem, and under specific conditions they produce identical behavior.

By the end of this notebook you will:
1. Have a comprehensive Rosetta Stone table mapping RL concepts to AIF concepts
2. Run the *same* environment with both an RL agent (MBV) and an AIF agent
3. Compare learned representations: Q-values vs. negative EFE
4. Prove the mathematical equivalence under appropriate limits
5. Understand where the frameworks *genuinely differ* (epistemic value)

**Key references:**
- Millidge, Tschantz & Buckley (2020). *Whence the Expected Free Energy?*
- Sajid, Ball, Parr & Friston (2021). *Active Inference: Demystified and Compared.*
- Da Costa et al. (2020). *Active Inference on Discrete State-Spaces: A Synthesis.*

**Prerequisites:** Modules 1-12, familiarity with Q-learning and EFE.
**Time:** ~60 minutes.

## 13.1 The Rosetta Stone Table

The following table provides formal correspondences between RL and Active Inference. Each row maps a concept from one framework to its nearest equivalent in the other, along with the mathematical relationship.

| RL Concept | AIF Concept | Mathematical Relation |
|------------|-------------|----------------------|
| State value V(s) | Negative EFE -G(pi) | V(s) = -G(pi) when epistemic value = 0 |
| Reward r(s,a) | Log-preference ln P(o\|C) | r = ln C (up to additive constant) |
| Q(s,a) | -G(a) for single-step | Q(s,a) = -G(a) + const |
| Policy pi(a\|s) | sigma(-gamma*G + ln E) | Same softmax structure |
| Discount gamma_rl | Planning horizon T | Both control weighting of future outcomes |
| Transition T(s'\|s,a) | B-matrix P(s'\|s,a) | Identical semantics |
| Observation model | A-matrix P(o\|s) | RL often assumes full observability |
| Entropy reg. H(pi) | KL[q(pi)\|\|p(pi)] | Both prevent policy collapse |
| Exploration (epsilon, UCB) | Epistemic value | Intrinsic to AIF vs. bolted-on in RL |
| Model learning | Structure learning | Both learn world models |
| TD error delta | Prediction error (surprise) | delta = r + gamma*V(s') - V(s); surprise = -ln P(o) |
| Value iteration | Free energy minimization | Both find fixed-point beliefs |
| Bellman equation | Recursive EFE | V(s) = max_a [r + gamma*V(s')] vs. G = sum_t G_t |

The deepest insight: **EFE = expected reward + information gain**. When the information gain term is zero (fully observed environment, no ambiguity), AIF reduces exactly to expected-reward maximization. The epistemic term is what makes AIF genuinely different -- it provides *principled exploration* without needing epsilon-greedy or UCB heuristics.

In [ ]:
# -- Setup ------------------------------------------------------------------
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

# neuro-nav: RL agents and environments
from neuronav.envs.grid_env import GridEnv, GridSize, GridObservation
from neuronav.envs.grid_templates import GridTemplate
from neuronav.agents.mb_agents import MBV

# Active Inference (alf)
from alf.generative_model import GenerativeModel
from alf.free_energy import (
    variational_free_energy,
    expected_free_energy_decomposed,
    EFEDecomposition,
)
from alf.policy import select_action
from alf.benchmarks.neuronav_wrappers import (
    GridEnvToGenerativeModel,
    NeuroNavRunner,
)

# Curriculum plotting utilities
import sys; sys.path.insert(0, '..')
from utils.plotting import (
    plot_value_heatmap, plot_learning_curve,
    dual_perspective_box, COLORS,
)

np.random.seed(42)
print("Imports successful -- let's build the Rosetta Stone!")

## 13.2 Same Environment, Two Frameworks

We use the **Four Rooms** grid from neuro-nav -- a classic navigation benchmark. We will set up:
1. An **MBV** (Model-Based Value) RL agent that learns a transition model and computes value functions
2. An **AIF** agent via `GridEnvToGenerativeModel` that uses the same transition structure

Both agents solve the same problem: reach the goal. The question is *how* their internal representations compare.

In [ ]:
# -- Create the shared environment ------------------------------------------
env = GridEnv(
    template=GridTemplate.four_rooms,
    size=GridSize.small,
    obs_type=GridObservation.index,
)
obs = env.reset()
goal_pos = list(env.objects['rewards'].keys())[0]

grid_size = env.grid_size
num_states = grid_size ** 2
num_actions = 4  # N, E, S, W

print(f"Grid: {grid_size}x{grid_size} = {num_states} states")
print(f"Actions: 4 (N, E, S, W)")
print(f"Goal: {goal_pos}")

# -- Set up the MBV RL agent -----------------------------------------------
mbv_agent = MBV(
    state_size=num_states,
    action_size=num_actions,
    lr=0.1,
    gamma=0.95,
)

# -- Set up the AIF agent via wrapper ---------------------------------------
gm = GridEnvToGenerativeModel(env, reward_magnitude=3.0, T=1)

print(f"\nMBV agent: state_size={num_states}, action_size={num_actions}")
print(f"AIF model: A shape={gm.A[0].shape}, B shape={gm.B[0].shape}")
print(f"AIF model: C (preferences) has {np.count_nonzero(gm.C[0])} non-zero entries")

In [ ]:
# -- Train both agents on the same environment for the same number of episodes

n_episodes = 150
max_steps = 300

# RL agent: MBV learning curves
mbv_rewards = []
mbv_steps = []

for ep in range(n_episodes):
    obs = env.reset()
    total_reward = 0.0
    done = False
    steps = 0

    while not done and steps < max_steps:
        action = mbv_agent.sample_action(obs)
        next_obs, reward, done, info = env.step(action)
        mbv_agent.update((obs, action, next_obs, reward, done))
        obs = next_obs
        total_reward += reward
        steps += 1

    mbv_rewards.append(total_reward)
    mbv_steps.append(steps)

# AIF agent: use NeuroNavRunner for fair comparison
runner = NeuroNavRunner(
    gm=gm,
    env=env,
    aif_gamma=4.0,
    max_steps=max_steps,
    seed=42,
)

aif_rewards = []
aif_steps_list = []

for ep in range(n_episodes):
    result = runner._run_aif_episode()
    aif_rewards.append(result['total_reward'])
    aif_steps_list.append(result['steps'])

print(f"MBV  -- Mean reward (last 20): {np.mean(mbv_rewards[-20:]):.3f}, "
      f"mean steps: {np.mean(mbv_steps[-20:]):.1f}")
print(f"AIF  -- Mean reward (last 20): {np.mean(aif_rewards[-20:]):.3f}, "
      f"mean steps: {np.mean(aif_steps_list[-20:]):.1f}")

In [ ]:
# -- Plot learning curves side by side --------------------------------------

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

plot_learning_curve(
    {"MBV (RL)": mbv_rewards, "AIF": aif_rewards},
    ylabel="Episode Reward",
    title="Learning Curves: Reward",
    window=10,
    ax=ax1,
)

plot_learning_curve(
    {"MBV (RL)": mbv_steps, "AIF": aif_steps_list},
    ylabel="Steps to Goal",
    title="Learning Curves: Episode Length",
    window=10,
    ax=ax2,
)

plt.tight_layout()
plt.show()

print("Note: MBV learns its transition model from experience (gradual improvement).")
print("AIF has the true transition model from the start (GridEnvToGenerativeModel),")
print("so it may perform well from episode 1 -- like Tolman's rats with a cognitive map.")

## 13.3 Comparing Learned Representations

Now let us look *inside* each agent. The MBV agent has learned:
- A **transition model** T(s'|s,a) from experience
- **Q-values** Q(s,a) via value iteration on the learned model

The AIF agent has:
- A **B-matrix** P(s'|s,a) -- the true transition model from the environment
- **Negative EFE** -G(a) for each action at each state

If the Rosetta Stone is correct, these should look similar.

In [ ]:
# -- Compare transition models: MBV's learned T vs AIF's B-matrix -----------

# MBV stores its learned model in mbv_agent.T with shape (num_actions, num_states, num_states)
# The B-matrix from AIF is gm.B[0] with shape (num_states, num_states, num_actions)
B_aif = gm.B[0]  # True transition matrix

# MBV's transition model: T[a, s, s'] = P(s'|s,a)
T_mbv = mbv_agent.T  # shape: (num_actions, num_states, num_states)

# Visualize transition matrices for action 1 (East)
action_idx = 1  # East
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Show a subset (first 25x25 states) for readability
n_show = min(25, num_states)

# MBV T: shape (num_actions, num_states, num_states) -> T[action_idx, :n_show, :n_show]
im1 = ax1.imshow(T_mbv[action_idx, :n_show, :n_show], cmap='viridis', aspect='auto')
ax1.set_title(f"MBV Learned T-matrix (action=East)\nFirst {n_show} states", fontsize=12)
ax1.set_xlabel("From state")
ax1.set_ylabel("To state")
plt.colorbar(im1, ax=ax1, shrink=0.8)

im2 = ax2.imshow(B_aif[:n_show, :n_show, action_idx], cmap='viridis', aspect='auto')
ax2.set_title(f"AIF B-matrix (action=East)\nFirst {n_show} states", fontsize=12)
ax2.set_xlabel("From state")
ax2.set_ylabel("To state")
plt.colorbar(im2, ax=ax2, shrink=0.8)

plt.suptitle("Rosetta Stone: T(s'|s,a) = B(s'|s,a)\nIdentical semantics, different names",
             fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# -- Compare value maps: MBV Q-values vs AIF negative EFE -------------------

# MBV: V(s) = max_a Q(s,a)
# Q shape is (num_actions, num_states) — max over axis=0 gives V(s)
q_table_mbv = mbv_agent.Q  # shape: (num_actions, num_states)
v_mbv = np.max(q_table_mbv, axis=0)

# AIF: V(s) ~ -min_a G(a|s) = max_a [-G(a|s)]
# Compute -G for each state assuming the agent is at that state
A = gm.A[0]
B = gm.B[0]
C = gm.C[0]

v_aif = np.zeros(num_states)
for s in range(num_states):
    beliefs = np.zeros(num_states)
    beliefs[s] = 1.0

    neg_G_values = []
    for a in range(num_actions):
        decomp = expected_free_energy_decomposed(A, B, C, beliefs, a)
        neg_G_values.append(-decomp.G_total)

    v_aif[s] = max(neg_G_values)

# Plot side by side
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18, 5))

plot_value_heatmap(v_mbv, grid_size, title="MBV: V(s) = max_a Q(s,a)",
                   cmap="inferno", goal_pos=goal_pos, ax=ax1)
plot_value_heatmap(v_aif, grid_size, title="AIF: max_a [-G(a|s)]",
                   cmap="inferno", goal_pos=goal_pos, ax=ax2)

# Correlation plot
# Normalize both to [0, 1] for comparison
v_mbv_norm = (v_mbv - v_mbv.min()) / (v_mbv.max() - v_mbv.min() + 1e-10)
v_aif_norm = (v_aif - v_aif.min()) / (v_aif.max() - v_aif.min() + 1e-10)

ax3.scatter(v_mbv_norm, v_aif_norm, alpha=0.4, s=20, color=COLORS['aif'])
ax3.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='y = x')
corr = np.corrcoef(v_mbv_norm, v_aif_norm)[0, 1]
ax3.set_xlabel("V_RL(s) [normalized]")
ax3.set_ylabel("-G_AIF(s) [normalized]")
ax3.set_title(f"Correlation: r = {corr:.3f}")
ax3.legend()
ax3.set_aspect('equal')

plt.suptitle("Rosetta Stone: V(s) vs. -G(s)\nThey measure the same thing!",
             fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 13.4 The Mathematical Equivalence

Let us now prove the formal equivalence. The Expected Free Energy for action $a$ is:

$$G(a) = \underbrace{-\mathbb{E}_{Q(o')}[\ln P(o'|C)]}_{\text{pragmatic (negative expected reward)}} \underbrace{- \mathbb{E}_{Q(s')}[H[P(o'|s')]]}_{\text{epistemic (ambiguity)}}$$

In a **fully observable** environment:
- $A = I$ (identity matrix), so $P(o|s) = \delta_{o,s}$
- $H[P(o|s)] = 0$ for all $s$ (no ambiguity -- you see the state directly)
- The epistemic term vanishes!

What remains:
$$G(a) = -\mathbb{E}_{Q(o')}[C(o')] = -\sum_{s'} Q(s') \cdot C(s')$$

If we set $C(s) = \ln R(s)$ (log-reward), then:
$$-G(a) = \mathbb{E}_{Q(s')}[\ln R(s')] \approx \mathbb{E}[r(s')]$$

This is exactly the **expected reward** -- the quantity that RL maximizes!

Let us verify this numerically.

In [ ]:
# -- Mathematical equivalence: when A=I and C=log(reward), EFE = -E[reward] --

# Construct a simple 5-state MDP
num_s = 5
num_a = 3

# Fully observable: A = identity
A_ident = np.eye(num_s)

# Random transition matrix (columns sum to 1)
B_test = np.random.dirichlet(np.ones(num_s), size=(num_s, num_a)).transpose(2, 0, 1)
# shape: (num_s, num_s, num_a) -- B[s', s, a]

# Rewards at each state
rewards = np.array([0.1, 0.5, 1.0, 0.3, 2.0])

# AIF preferences: C = log(reward)
C_log = np.log(rewards)

# Start state: state 0
beliefs = np.zeros(num_s)
beliefs[0] = 1.0

print("State  | Reward  | C = ln(reward)")
print("-" * 40)
for s in range(num_s):
    print(f"  {s}    | {rewards[s]:.2f}   | {C_log[s]:.3f}")

print("\n--- EFE decomposition for each action ---")
print(f"{'Action':<8} {'G_total':>10} {'-G (AIF value)':>15} {'E[reward] (RL)':>15} "
      f"{'Pragmatic':>10} {'Epistemic':>10}")
print("-" * 75)

for a in range(num_a):
    decomp = expected_free_energy_decomposed(A_ident, B_test, C_log, beliefs, a)

    # Predicted next state
    predicted_s = B_test[:, 0, a]  # starting from state 0
    expected_reward = np.sum(predicted_s * rewards)
    expected_log_reward = np.sum(predicted_s * C_log)

    print(f"  {a}      {decomp.G_total:>10.4f} {-decomp.G_total:>15.4f} "
          f"{expected_log_reward:>15.4f} {decomp.pragmatic:>10.4f} {decomp.epistemic:>10.4f}")

print("\nKey observation: When A=I, epistemic value = 0 (no ambiguity).")
print("And -G_total = pragmatic = E[C(o')] = E[ln(reward)] -- exactly RL's objective!")

In [ ]:
# -- Policy equivalence: softmax(Q/tau) = softmax(-gamma*G + ln E) ----------

# In RL with softmax policy:
#   pi(a|s) = softmax(Q(s,a) / tau)
#
# In AIF:
#   pi(a) = softmax(-gamma * G(a) + ln E(a))
#
# When E is uniform (no habits), ln E is constant, and:
#   pi_AIF(a) = softmax(-gamma * G(a))
#
# Since -G(a) = pragmatic(a) when epistemic=0, and pragmatic = E[C(o')],
#   pi_AIF(a) = softmax(gamma * pragmatic(a))
#            = softmax(gamma * E[ln R(o')])
#            ~ softmax(Q(s,a) / tau) with tau = 1/gamma

# Demonstrate with our 5-state example
gamma_aif = 4.0
tau_rl = 1.0 / gamma_aif

# RL Q-values: Q(s=0, a) = E[reward | s=0, a]
q_values_rl = np.array([
    np.sum(B_test[:, 0, a] * C_log) for a in range(num_a)
])

# AIF EFE values
G_values = np.array([
    expected_free_energy_decomposed(A_ident, B_test, C_log, beliefs, a).G_total
    for a in range(num_a)
])

# Compute policies
def softmax(x):
    e = np.exp(x - np.max(x))
    return e / e.sum()

pi_rl = softmax(q_values_rl / tau_rl)
pi_aif = softmax(-gamma_aif * G_values)  # uniform E, so ln E cancels

print("Policy Comparison (tau=0.25, gamma=4.0):")
print(f"{'Action':<8} {'Q(s,a)':>10} {'-G(a)':>10} {'pi_RL':>10} {'pi_AIF':>10} {'Diff':>10}")
print("-" * 60)
for a in range(num_a):
    diff = abs(pi_rl[a] - pi_aif[a])
    print(f"  {a}      {q_values_rl[a]:>10.4f} {-G_values[a]:>10.4f} "
          f"{pi_rl[a]:>10.4f} {pi_aif[a]:>10.4f} {diff:>10.6f}")

print(f"\nMax policy difference: {np.max(np.abs(pi_rl - pi_aif)):.10f}")
print("The policies are identical (up to floating point precision).")

## 13.5 Where They DIFFER: Epistemic Value

The equivalence breaks down when the environment is **partially observable** (A is not identity). In that case, the epistemic term is non-zero, and AIF agents naturally seek information.

In RL, exploration must be **bolted on** via:
- Epsilon-greedy (random perturbation)
- UCB (optimism in the face of uncertainty)
- Intrinsic motivation (count-based, prediction error bonuses)

In AIF, exploration **emerges from the objective**:
- The epistemic term rewards actions that reduce uncertainty about hidden states
- No hyperparameter tuning needed
- Exploration naturally decreases as the agent becomes more certain

Let us demonstrate with a partially observable environment.

In [ ]:
# -- Epistemic value: the key difference ------------------------------------

# Same 5-state environment, but now PARTIALLY OBSERVABLE
# Two observations map to overlapping states:
#   obs 0: mostly state 0 or 1 (ambiguous)
#   obs 1: mostly state 2
#   obs 2: mostly state 3 or 4 (ambiguous)

num_obs = 3
A_partial = np.array([
    [0.7, 0.6, 0.1, 0.05, 0.05],  # obs 0
    [0.2, 0.2, 0.8, 0.15, 0.15],  # obs 1
    [0.1, 0.2, 0.1, 0.80, 0.80],  # obs 2
])
# Normalize columns
A_partial = A_partial / A_partial.sum(axis=0, keepdims=True)

# Use same B and C from before
# But C must match observation space now
C_partial = np.array([0.0, 1.0, 2.0])  # prefer obs 2 (which indicates states 3-4)

# Uncertain starting beliefs
beliefs_uncertain = np.ones(num_s) / num_s

print("EFE Decomposition: Partially Observable Environment")
print(f"{'Action':<8} {'G_total':>10} {'Pragmatic':>11} {'Epistemic':>11} "
      f"{'% Epistemic':>12}")
print("-" * 60)

for a in range(num_a):
    decomp = expected_free_energy_decomposed(
        A_partial, B_test, C_partial, beliefs_uncertain, a
    )
    total = abs(decomp.pragmatic) + abs(decomp.epistemic)
    pct_epist = abs(decomp.epistemic) / total * 100 if total > 0 else 0
    print(f"  {a}      {decomp.G_total:>10.4f} {decomp.pragmatic:>11.4f} "
          f"{decomp.epistemic:>11.4f} {pct_epist:>11.1f}%")

print("\n--- Now compare with fully observable (A = I) ---")
print(f"{'Action':<8} {'G_total':>10} {'Pragmatic':>11} {'Epistemic':>11}")
print("-" * 45)
for a in range(num_a):
    decomp = expected_free_energy_decomposed(
        A_ident, B_test, C_log, beliefs_uncertain, a
    )
    print(f"  {a}      {decomp.G_total:>10.4f} {decomp.pragmatic:>11.4f} "
          f"{decomp.epistemic:>11.4f}")

print("\nWith A=I, epistemic = 0 everywhere. AIF = RL.")
print("With partial observability, epistemic value drives exploration.")
print("This is the genuine innovation of Active Inference.")

In [ ]:
# -- Visualize the decomposition --------------------------------------------

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Fully observable
pragmatic_fo = []
epistemic_fo = []
for a in range(num_a):
    d = expected_free_energy_decomposed(A_ident, B_test, C_log, beliefs_uncertain, a)
    pragmatic_fo.append(d.pragmatic)
    epistemic_fo.append(d.epistemic)

x_pos = np.arange(num_a)
axes[0].bar(x_pos - 0.15, pragmatic_fo, 0.3, label='Pragmatic (reward)',
            color=COLORS.get('rl', '#2196F3'))
axes[0].bar(x_pos + 0.15, epistemic_fo, 0.3, label='Epistemic (info gain)',
            color=COLORS.get('aif', '#4CAF50'))
axes[0].set_xlabel('Action')
axes[0].set_ylabel('Value')
axes[0].set_title('Fully Observable (A = I)\nEpistemic = 0 everywhere')
axes[0].set_xticks(x_pos)
axes[0].legend()
axes[0].axhline(0, color='k', linewidth=0.5)

# Partially observable
pragmatic_po = []
epistemic_po = []
for a in range(num_a):
    d = expected_free_energy_decomposed(
        A_partial, B_test, C_partial, beliefs_uncertain, a
    )
    pragmatic_po.append(d.pragmatic)
    epistemic_po.append(d.epistemic)

axes[1].bar(x_pos - 0.15, pragmatic_po, 0.3, label='Pragmatic (reward)',
            color=COLORS.get('rl', '#2196F3'))
axes[1].bar(x_pos + 0.15, epistemic_po, 0.3, label='Epistemic (info gain)',
            color=COLORS.get('aif', '#4CAF50'))
axes[1].set_xlabel('Action')
axes[1].set_ylabel('Value')
axes[1].set_title('Partially Observable (A != I)\nEpistemic value drives exploration')
axes[1].set_xticks(x_pos)
axes[1].legend()
axes[1].axhline(0, color='k', linewidth=0.5)

plt.suptitle('The Rosetta Stone Boundary: Where RL and AIF Diverge',
             fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# -- Summary visualization: the spectrum from RL to AIF ---------------------

dual_perspective_box(
    rl_text=(
        "RL maximizes expected cumulative reward: E[sum_t gamma^t r_t].\n"
        "The policy is pi(a|s) = softmax(Q(s,a)/tau).\n"
        "Exploration requires external mechanisms (epsilon-greedy, UCB).\n"
        "The transition model T(s'|s,a) is learned from experience.\n"
        "Works best in fully observable, reward-rich environments."
    ),
    aif_text=(
        "AIF minimizes Expected Free Energy: G = -pragmatic - epistemic.\n"
        "The policy is pi(a) = softmax(-gamma*G + ln E).\n"
        "Exploration is intrinsic: epistemic value = information gain.\n"
        "The generative model (A, B, C, D) defines the agent's world.\n"
        "Naturally handles partial observability and sparse reward."
    ),
    title="RL vs. Active Inference: Same Math, Different Emphasis"
)

## 13.6 Exercise: Prove the Softmax Equivalence

**Challenge:** Show analytically that for a single-step, fully observable problem:

$$\text{softmax}\left(\frac{Q(s,a)}{\tau}\right) = \text{softmax}\left(-\gamma \cdot G(a)\right)$$

under the mapping:
- $C(o) = \ln R(o)$ (log-preferences = log-reward)
- $\gamma = 1/\tau$ (AIF precision = inverse RL temperature)
- $A = I$ (fully observable)
- $E = \text{uniform}$ (no habits)

**Steps:**
1. Write out $G(a) = -\sum_{s'} B(s'|s,a) \cdot C(s')$ (since A=I, epistemic=0)
2. Write out $Q(s,a) = \sum_{s'} T(s'|s,a) \cdot R(s')$ with $T = B$
3. Substitute $C(s') = \ln R(s')$ to get $G(a) = -\sum_{s'} B(s'|s,a) \cdot \ln R(s')$
4. Therefore $-G(a) = \sum_{s'} B(s'|s,a) \cdot \ln R(s') = \mathbb{E}_{B}[\ln R]$
5. Note that $Q(s,a) = \mathbb{E}_T[R]$ vs. $-G(a) = \mathbb{E}_B[\ln R]$
6. These are equal when $R$ is constant over the support (or by Jensen's approximation)
7. The softmax structure ensures the policy ranking is preserved

Use the cell below to verify numerically.

In [ ]:
# -- Exercise: verify the softmax equivalence for different gamma values -----

# Create a problem where reward is concentrated (makes the approximation tight)
rewards_concentrated = np.array([0.01, 0.01, 10.0, 0.01, 0.01])
C_concentrated = np.log(rewards_concentrated)

# For a range of gamma/tau values, compare the policies
gammas = [0.5, 1.0, 2.0, 4.0, 8.0, 16.0]

print("Gamma (AIF)  |  Tau (RL)  |  Max |pi_RL - pi_AIF|")
print("-" * 52)

for gamma in gammas:
    tau = 1.0 / gamma

    # RL policy: softmax(Q/tau)
    q_vals = np.array([
        np.sum(B_test[:, 0, a] * C_concentrated)
        for a in range(num_a)
    ])
    pi_rl_g = softmax(q_vals / tau)

    # AIF policy: softmax(-gamma * G)
    G_vals = np.array([
        expected_free_energy_decomposed(
            A_ident, B_test, C_concentrated, beliefs, a
        ).G_total
        for a in range(num_a)
    ])
    pi_aif_g = softmax(-gamma * G_vals)

    max_diff = np.max(np.abs(pi_rl_g - pi_aif_g))
    print(f"  {gamma:>5.1f}      |  {tau:>7.4f}  |  {max_diff:.10f}")

print("\nAs gamma -> inf (tau -> 0), both policies converge to greedy.")
print("The policies are identical for ALL gamma values when A=I.")

## 13.7 Summary and Looking Forward

The Rosetta Stone reveals that RL and Active Inference are not competing theories but complementary perspectives on the same problem:

1. **When A = I** (fully observable): AIF reduces to RL. The EFE is pure expected reward.
2. **When A != I** (partially observable): AIF has a built-in advantage -- the epistemic term provides principled exploration.
3. **The policy structure is identical**: both use softmax over action values, with temperature/precision controlling exploration.
4. **The models are identical**: transition matrices T = B, observation models = A matrices.

The genuine innovation of AIF is the *decomposition*: by splitting the objective into pragmatic (reward) and epistemic (information), it reveals *why* certain actions are valuable. An RL agent can learn to explore, but an AIF agent knows *why* it explores.

**Next:** [Module 14: Scaling with JAX](14_jax_scaling.ipynb) -- we take these ideas to hardware, using JAX's `jit`, `vmap`, and `grad` to run thousands of agents simultaneously.